In [1306]:
####################################
#ENVIRONMENT SETUP

In [1307]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [1308]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [1309]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [1310]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [1311]:
####################################
#LOADING CLASSES

In [1312]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="UpdraftArea",
                                dtype='float32')

=== CM1 Data Summary ===
 Simulation #:   4
 Resolution:     1km
 Time step:      3min
 Vertical levels:95
 Parcels:        20e6
 Data file:      /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Simulation_Four/cm1out_000001.nc
 Parcel file:    /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Model/cm1r20.3/run/MODEL_OUTPUT/Simulation_Four/cm1out_pdata_1km_3min_20e6np.nc
 Time steps:     221

=== DataManager Summary ===
 inputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData
 outputDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/CalculateMoreVariables
 inputDataDirectory #:   /mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/TimeSplitModelData/Simulation_4_1km_3min_95nz/ModelData
 inputParcelDirec

In [1313]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

Running timesteps from 0:7 



In [1314]:
####################################
#FUNCTIONS

In [1315]:
#Variable Loading Functions

def GetVarNames():
    return ['A_g','A_c']

def GetInputVariables(inputDataDirectory, timeString, varNames):
    inputDictionary = {varName: CallVariable(ModelData, DataManager, timeString, varName) for varName in varNames}
    return inputDictionary

In [1316]:
#Updraft Area Calculation Functions

def CalculateBinaryArea_V1(A, shapeApproximation="oval"): 
    """
    For each updraft point, calculate area as the true grid cell count
    of the 2D connected region it belongs to.

    NOT RECOMMENDED SINCE LARGE CONTIGUOUS REGIONS COUNTNED AS GIGANTIC UPDRAFT
    """
    [Nz, Ny, Nx] = A.shape
    updraftArea = np.full(A.shape, np.nan)

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue

        labeled, n_features = label(slice2D)
        for i in range(1, n_features + 1):
            mask = labeled == i
            updraftArea[k][mask] = mask.sum()  # grid cell count, not x*y

    updraftArea *= ModelData.dx * ModelData.dy
    if shapeApproximation == "oval":
        updraftArea *= (np.pi / 4)

    return updraftArea, [], []

def LabelSizesOfIndividualUpdrafts(array1D):
    """Label connected regions and return per-cell sizes."""
    labeled, _ = label(array1D)
    sizes = np.bincount(labeled.ravel())
    return labeled, sizes

def ApplyLengths(array1D, sizes,labeled):
    return np.where(array1D, sizes[labeled], np.nan)    

from scipy.ndimage import label
def CalculateBinaryLength(A, dimension, BC_x='open', BC_y='periodic'):
    """
    For each updraft point, calculate the length of the contiguous 
    updraft run it belongs to, along the specified dimension.
    """

    #Get Dimensional Information
    [Nz, Ny, Nx] = A.shape; length = np.full(A.shape, np.nan)
    [N, N_loop, BC] = (Nx, Ny, BC_x) if dimension == 'x' else (Ny, Nx, BC_y)

    #Looping
    for k in range(Nz): #Looping over all levels
        for n in range(N_loop): #Looping over each dimensional slice
            array1D = A[k, n, :] if dimension == 'x' else A[k, :, n] #Indexing corresponding 1d array
            if array1D.sum() == 0: continue
            
            if BC == 'periodic': #if boundary conditions are periodic, meaning updrafts continue to the other side past the boundary
                array1D = np.tile(array1D, 3)
                labeled, sizes = LabelSizesOfIndividualUpdrafts(array1D=array1D)
                lengthWhere = ApplyLengths(array1D=array1D, sizes=sizes, labeled=labeled)
                lengthWhere = np.minimum(lengthWhere[N:2*N], N)
            elif BC == 'open': #if boundary conditions are open (e.g. radiative), meaning updrafts disappear at the boundary
                labeled, sizes = LabelSizesOfIndividualUpdrafts(array1D=array1D)
                lengthWhere = ApplyLengths(array1D=array1D, sizes=sizes, labeled=labeled)

            if dimension == 'x':
                length[k, n, :] = lengthWhere
            elif dimension == 'y':
                length[k, :, n] = lengthWhere
    return length

def CalculateBinaryArea_V2(A,
                        shapeApproximation="oval"):
    """
    For each updraft point, calculate the local cross-sectional area
    as xLength * yLength.

    NOT RECOMMENDED SINCE UPDRAFT AREA COULD BE LARGE, BUT VERY CLOSE TO EDGE OF CLOUD
    """
    xLength = CalculateBinaryLength(A, dimension='x')
    yLength = CalculateBinaryLength(A, dimension='y')

    #calculating updraft area
    updraftArea = xLength * yLength
    #converting from grid index to meters
    updraftArea *= ModelData.dx * ModelData.dy
    #converting to oval approximation
    if shapeApproximation=="oval":
        updraftArea *= (np.pi / 4)
    
    return updraftArea, xLength,yLength

def CalculateBinaryArea_V3(A, shapeApproximation="oval"):
    """
    For each updraft point, calculate area as mean_x_length * mean_y_length
    for the 2D connected region it belongs to.

    #RECOMMENDED
    """
    [Nz, Ny, Nx] = A.shape
    area = np.full(A.shape, np.nan)
    xLength_out = np.full(A.shape, np.nan)
    yLength_out = np.full(A.shape, np.nan)

    # Get per-point x and y lengths
    xLength = CalculateBinaryLength(A, dimension='x')
    yLength = CalculateBinaryLength(A, dimension='y')

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue

        labeled, n_features = label(slice2D)

        for i in range(1, n_features + 1):
            mask = labeled == i
            xLength_out[k][mask] = np.nanmean(xLength[k][mask])
            yLength_out[k][mask] = np.nanmean(yLength[k][mask])

    #calculating updraft area
    updraftArea = xLength_out * yLength_out
    #converting from grid index to meters
    updraftArea *= ModelData.dx * ModelData.dy
    #converting to oval approximation
    if shapeApproximation == "oval":
        updraftArea *= (np.pi / 4)

    return updraftArea, xLength_out, yLength_out

from scipy.ndimage import distance_transform_edt
def CalculateEuclideanDistanceFromNearestEdge(A,dy=ModelData.dy,dx=ModelData.dx):
    """
    For each updraft point, calculate the Euclidean distance to the 
    nearest non-updraft cell.
    """
    [Nz, Ny, Nx] = A.shape
    distance = np.full(A.shape, np.nan)

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue
        # sampling gives physical distance in meters
        distance[k] = np.where(slice2D, distance_transform_edt(slice2D, sampling=(dy,dx)), np.nan)

    return distance

In [1317]:
#Running Functions

def VariableCalculation(inputDictionary,varNames):
    [A_g,A_c] = (inputDictionary[k] for k in varNames)

    [updraftArea_g,_,_] = CalculateBinaryArea_V3(A_g)
    [updraftArea_c,_,_] = CalculateBinaryArea_V3(A_c)

    updraftEdgeDistance_g = CalculateEuclideanDistanceFromNearestEdge(A_g)
    updraftEdgeDistance_c = CalculateEuclideanDistanceFromNearestEdge(A_c)
    
    outputDictionary={'updraftArea_g': updraftArea_g,
                      'updraftArea_c': updraftArea_c,
                      'updraftEdgeDistance_g': updraftEdgeDistance_g,
                      'updraftEdgeDistance_c': updraftEdgeDistance_c}
    
    return outputDictionary

def RunCode():
    for t in tqdm(loop_elements, total=len(loop_elements)):
        if np.mod(t,1)==0: print(f'Current time {t}')   

        #get variable names
        varNames = GetVarNames()
    
        #loading input variables
        timeString = ModelData.timeStrings[t]
        inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
    
        #calculating variables
        outputDictionary = VariableCalculation(inputDictionary,varNames)
        
        #outputting
        DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary)

In [1318]:
####################################
#RUNNING

In [ ]:
#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
RunCode()

In [ ]:
####################################
#PLOTTING 

# Updraft Threshold Fields and Updraft Area Calculations look (Case Study)

In [ ]:
#Plotting Functions

def PlotSlice(data, mask, xLevel=100, yLevels=np.arange(0,200), varType='symmetric',
              yLim=None, numLevels=100,levels=None,zorder=None,
              axis=None,cbarLabel='w (m/s)'):

    if axis is None:
        fig,axis=plt.subplots()

    cmap='RdBu_r' if varType == 'symmetric' else 'turbo'

    #setting z limit
    zHeight=6
    zIndex = np.abs(ModelData.zh-zHeight).argmin()
    data_2=data.copy()
    data_2[zIndex+1:]=np.nan
    axis.set_ylim(0, zHeight) if yLim is None else axis.set_ylim(yLim)
    
    data_plot = np.where(mask[:, yLevels, xLevel], data_2[:, yLevels, xLevel], np.nan)

    if levels is None:
        if varType == 'symmetric':
            vmax=np.nanmax(data_plot) 
            levels = np.linspace(-vmax,vmax,numLevels)
        elif varType == 'positive':
            levels=numLevels



    cf=axis.contourf(ModelData.yh[yLevels],ModelData.zh,data_plot,cmap=cmap,levels=levels,zorder=zorder)
    plt.colorbar(cf, ax=axis,label=cbarLabel)

    axis.set_ylabel('z (km)'); axis.set_xlabel('y (km)'); 
    
    
def Plot_XYSlice(axis):
    #Finding Ideal Cloud
    
    #Simulation
    #Simulation 4
    
    #Height
    zHeight=2.01
    zIndex = np.abs(ModelData.zh-zHeight).argmin()
    
    #Cloud
    a=qcqi[zIndex].copy()
    # a[(a<=1e-6)]=np.nan
    cf=axis.contourf(ModelData.xh,ModelData.yh,a,cmap='turbo',levels=16,zorder=0);plt.colorbar(cf,ax=axis,label='qc+qi (kg/kg)')
    axis.contour(a>1e-6,colors='red',zorder=2)
    
    #Updraft
    a=w[zIndex].copy()
    a[(a<=0.1)&(a>=-0.1)]=np.nan
    axis.contour(ModelData.xh,ModelData.yh,a,colors='white',levels=16,zorder=3)
    
    #Refining XLim
    yLevels=np.arange(90,105+1)
    axis.set_ylim(yLevels[0],yLevels[-1])
    step=5
    axis.set_xlim(180-step,185+step)
    
    
    #==>
    xLevel=183
    yLevel=98
    axis.axvline(xLevel,color='black',linewidth=1)
    axis.axhline(yLevel,color='black',linewidth=1)
    
    axis.set_xlabel('x (km)'); axis.set_ylabel('y (km)')
    axis.set_title('qc+qi (contour): qc+qi > 1e-6 (red)\nw (lines): w>0.1 - and w<-0.1 -- (white)');

    return zIndex,yLevels,xLevel

def Plot_ZYSlice(axis):

    updraft = (w==w) #disabled
    PlotSlice(data=w,mask=updraft, xLevel=xLevel, yLevels=yLevels,varType='positive',zorder=0,axis=axis,cbarLabel='w (m/s)')
    cf=axis.contour(ModelData.yh[yLevels],ModelData.zh,qcqi[:,yLevels,xLevel]>1e-6,colors='red',zorder=2)
    
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.1,colors='white',linewidths=1,zorder=3)
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.5,colors='white',linestyles='dashed',linewidths=1,zorder=3)
    axis.axvline(yLevel,color='black',linewidth=2,zorder=1)
    axis.axhline(zHeight,color='black',linewidth=2,zorder=1)

    axis.set_title('w (contour): qc+qi > 1e-6 (red)\nw (lines): w>0.1 - and w>0.5 -- (white)');

def Plot_ZYSlice_VarData(axis,varData,cbarLabel="",levels=np.arange(0,10+1)):
    
    updraft = (varData==varData) #disabled
    PlotSlice(data=varData,mask=updraft, xLevel=xLevel, yLevels=yLevels,varType='positive',zorder=0,levels=levels,
              axis=axis,cbarLabel=cbarLabel)
    axis.contour(ModelData.yh[yLevels],ModelData.zh,qcqi[:,yLevels,xLevel]>1e-6,colors='red',zorder=2)
    
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.1,colors='white',linewidths=1,zorder=3)
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.5,colors='white',linestyles='dashed',
                 linewidths=1,zorder=3)
    axis.axvline(yLevel,color='black',linewidth=2,zorder=1)
    axis.axhline(zHeight,color='black',linewidth=2,zorder=1)

In [1366]:
#Calculating And Loading Data

t=200
timeString = ModelData.timeStrings[t]

#Calculate Updraft Area
if "outputDictionary" not in globals():
    #get variable names
    varNames = GetVarNames()
    #loading input variables
    inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
    #calculating variables
    outputDictionary = VariableCalculation(inputDictionary,varNames)
    updraftEdgeDistance_g=outputDictionary['updraftEdgeDistance_g']/1e3
    updraftEdgeDistance_c=outputDictionary['updraftEdgeDistance_c']/1e3
    updraftArea_g=outputDictionary['updraftArea_g']
    updraftArea_c=outputDictionary['updraftArea_c']
    
    updraftArea_g=np.sqrt(updraftArea_g/1e6)
    updraftArea_c=np.sqrt(updraftArea_c/1e6)

    #making it so contourf works for plotting
    updraftArea_g[np.isnan(updraftArea_g)]=0 
    updraftArea_c[np.isnan(updraftArea_c)]=0
    updraftEdgeDistance_g[np.isnan(updraftEdgeDistance_g)]=0
    updraftEdgeDistance_c[np.isnan(updraftEdgeDistance_c)]=0

#Calculate Other Variables
varNames = GetVarNames()+['qcqi','winterp','buoyancy2']
inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
    
[A_g,A_c,qcqi,w,B] = (inputDictionary[k] for k in varNames)

In [ ]:
#Plotting
fig,axes=plt.subplots(3,2,figsize=(16,12))
axes=axes.ravel()

[zIndex,yLevels,xLevel] = Plot_XYSlice(axis=axes[0])
Plot_ZYSlice(axis=axes[1])
Plot_ZYSlice_VarData(axes[2],updraftArea_g,levels=np.arange(0,10+1),cbarLabel="sqrt( updraftArea_g ) (km)")
Plot_ZYSlice_VarData(axes[3],updraftArea_c,levels=np.arange(0,10+1),cbarLabel="sqrt( updraftArea_c ) (km)")
Plot_ZYSlice_VarData(axes[4],updraftEdgeDistance_g,levels=15,cbarLabel="updraftEdgeDistance_g (km)")
Plot_ZYSlice_VarData(axes[5],updraftEdgeDistance_c,levels=15,cbarLabel="updraftEdgeDistance_c (km)")

In [1319]:
####################################
#TESTING

In [ ]:
# #testing Showcasing V1 vs V3 algorithm differences

# #Calculating
# [updraftArea_simple,_,_] = CalculateBinaryArea_V1(A_c)
# [updraftArea,_,_] = CalculateBinaryArea_V3(A_c)
# [_,lengthX,lengthY] = CalculateBinaryArea_V2(A_c)

# updraftArea_simple[np.isnan(updraftArea_simple)]=0;updraftArea[np.isnan(updraftArea)]=0; lengthX[np.isnan(lengthX)]=0; lengthY[np.isnan(lengthY)]=0
# updraftArea_simple/=1e6;updraftArea/=1e6
# updraftArea_simple=np.sqrt(updraftArea_simple); updraftArea=np.sqrt(updraftArea)


# #Plotting
# fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# # Row 0: V1 (col 0), rest blank
# axis = axes[0,0]
# a = updraftArea_simple[zIndex, yLevels, 175:190]
# cf = axis.contourf(a, levels=7)
# plt.colorbar(cf, ax=axis, label='sqrt( updraftArea_c ) (km)')
# axis.set_title('V1: area = total connected-region size\n(same value for every cell in region)')
# axis.set_ylabel('y (index)'); axis.set_xlabel('x (index)')

# axes[0,1].axis('off')
# axes[0,2].axis('off')

# # Row 1: V3 (all 3 columns)
# axis = axes[1,0]
# a = lengthX[zIndex, yLevels, 175:190]
# cf = axis.contourf(a, levels=np.arange(0, 5+1, 1))
# plt.colorbar(cf, ax=axis, label='Length_X (km)')
# axis.set_title('V3: x-extent of updraft at each point')
# axis.set_ylabel('y (index)'); axis.set_xlabel('x (index)')

# axis = axes[1,1]
# a = lengthY[zIndex, yLevels, 175:190]
# cf = axis.contourf(a, levels=np.arange(0, 7+1, 1))
# plt.colorbar(cf, ax=axis, label='Length_Y (km)')
# axis.set_title('V3: y-extent of updraft at each point')
# axis.set_ylabel('y (index)'); axis.set_xlabel('x (index)')

# axis = axes[1,2]
# a = updraftArea[zIndex, yLevels, 175:190]
# cf = axis.contourf(a, levels=7)
# plt.colorbar(cf, ax=axis, label='sqrt( updraftArea_c ) (km)')
# axis.set_title('V3: area = mean(Length_X) × mean(Length_Y)\n(same value for every cell in region)')
# axis.set_ylabel('y (index)'); axis.set_xlabel('x (index)')

# fig.suptitle('Comparing updraft area methods: V1 (Sum of Areas) vs V3 (Average of Local Grid Object Lengths)', fontsize=13)
# plt.tight_layout()

In [1383]:
# #testing, are general updrafts seperated enough or too merged and needs better conditions to find BL thermals?

# varNames = GetVarNames()
# [A_g,A_c] = (inputDictionary[k] for k in varNames)

# maxZ=24
# zLevels = np.arange(0, maxZ, 8)
# fig, axes = plt.subplots(len(zLevels), 2, figsize=(12, 5*len(zLevels)))

# for i, z in enumerate(tqdm(zLevels)):
#     #Plotting A_g
#     axis = axes[i,0]
#     cf = axis.contourf(A_g[z, ySlice, xSlice], levels=[-0.5,0.5,1.5], colors=['white','black'])
#     plt.colorbar(cf, ax=axis)
#     axis.set_title(f'A_g (w > 0.1), z={ModelData.zh[z]*1e3:.0f} m')

#     #How many updrafts are interconnected?
#     axis = axes[i,1]
#     labeled, n_features = label(A_g[z, ySlice, xSlice])
#     rng = np.random.permutation(n_features) + 1
#     remap = np.zeros(n_features+1, dtype=int)
#     remap[1:] = rng
#     labeled = remap[labeled].astype(float)
#     labeled[labeled==0] = np.nan
#     cmap = plt.colormaps['nipy_spectral'].resampled(n_features+1)
#     cf = axis.pcolormesh(labeled, cmap=cmap, vmin=-0.5, vmax=n_features+0.5)
#     plt.colorbar(cf, ax=axis, label='Label')
#     axis.set_title(f'{n_features} individual updrafts, z={ModelData.zh[z]*1e3:.0f} m')

# plt.tight_layout()
# plt.show()

In [1382]:
# #testing, trying to use max_filter and watershed to solve issue

# from scipy.ndimage import label, maximum_filter
# from skimage.segmentation import watershed
# # def GetLabeled():
# #     mask = (w > 0.1) & (qcqi < 1e-6)
    
# #     # find local maxima of w within the updraft mask
# #     local_max = (w == maximum_filter(w, size=5)) & mask
# #     markers, _ = label(local_max)
    
# #     # watershed splits the connected mask using w as the "elevation" to climb down from peaks
# #     labeled = watershed(-w, markers=markers, mask=mask)

# #     return labeled
# def GetLabeled(z):
#     mask = (w[z, ySlice, xSlice] > 0.1) & (qcqi[z, ySlice, xSlice] < 1e-6)
#     w_slice = w[z, ySlice, xSlice]
    
#     local_max = (w_slice == maximum_filter(w_slice, size=5)) & mask
#     markers, _ = label(local_max)
    
#     labeled = watershed(-w_slice, markers=markers, mask=mask)
#     n_features = labeled.max()
#     return labeled, n_features

# varNames = GetVarNames()
# [A_g,A_c] = (inputDictionary[k] for k in varNames)

# maxZ=24
# zLevels = np.arange(0, maxZ, 8)
# fig, axes = plt.subplots(len(zLevels), 2, figsize=(12, 5*len(zLevels)))

# for i, z in enumerate(tqdm(zLevels)):
#     #Plotting A_g
#     axis = axes[i,0]
#     cf = axis.contourf(A_g[z, ySlice, xSlice], levels=[-0.5,0.5,1.5], colors=['white','black'])
#     plt.colorbar(cf, ax=axis)
#     axis.set_title(f'A_g (w > 0.1), z={ModelData.zh[z]*1e3:.0f} m')

#     #How many updrafts are interconnected?
#     axis = axes[i,1]



    
#     labeled, n_features = GetLabeled(z)
#     rng = np.random.permutation(n_features) + 1
#     remap = np.zeros(n_features+1, dtype=int)
#     remap[1:] = rng
#     labeled = remap[labeled].astype(float)
#     labeled[labeled==0] = np.nan
#     cmap = plt.colormaps['nipy_spectral'].resampled(n_features+1)
#     cf = axis.pcolormesh(labeled, cmap=cmap, vmin=-0.5, vmax=n_features+0.5)
#     plt.colorbar(cf, ax=axis, label='Label')
#     axis.set_title(f'{n_features} individual updrafts, z={ModelData.zh[z]*1e3:.0f} m')

# plt.tight_layout()
# plt.show()